In [62]:
import json
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

In [63]:
MODEL_ID="llama3.1:8b"

client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

In [64]:
class WeatherArgs(BaseModel):
    city: str = Field(description="City name, e.g. Chicago")
    
class ExchangeRateArgs(BaseModel):
    base_currency: str = Field(description="3 letter currency code, e.g. USD")
    target_currency: str = Field(description="3 letter currency code, e.g. EUR")

In [65]:
def get_weather(args: WeatherArgs) -> dict:
    fake = {
        "Chicago": "34F and windy", 
        "Bishkek": "55F and shiny",
        "Tokyo": "32F and rainy",
        "Naryn": "10F and snow",
    }
    return {"city": args.city, "weather": fake[args.city]}

def get_exchange_rate(args: ExchangeRateArgs) -> dict:
    return {
        "base": args.base_currency.upper(),
        "target": args.target_currency.upper(),
        "rate": 0.92
    }

In [66]:
# Test functions
city = WeatherArgs(city="Chicago")
print(get_weather(city))

exchange_rate_1 = ExchangeRateArgs(base_currency="KGS", target_currency="USD")
print(get_exchange_rate(exchange_rate_1))

{'city': 'Chicago', 'weather': '34F and windy'}
{'base': 'KGS', 'target': 'USD', 'rate': 0.92}


In [67]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city.",
            "parameters": WeatherArgs.model_json_schema(),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get conversion rate between 2 currencies.",
            "parameters": ExchangeRateArgs.model_json_schema(),
        },
    },
]

tool_dispatch = {
    "get_weather": lambda raw_args: get_weather(WeatherArgs.model_validate(raw_args)),
    "get_exchange_rate": lambda raw_args: get_exchange_rate(ExchangeRateArgs.model_validate(raw_args))
}

In [79]:
def ask_with_tool(prompt: str, model: str = "gpt-4o") -> str:
    messages = [{"role": "user", "content": prompt}]
    
    while True:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        message = response.choices[0].message
        print("#"*50)
        print(message)
        
        if not message.tool_calls:
            return message.content or ""
        
        # Save the assistant's tool-call message first
        messages.append(message.model_dump())
        print(messages)
        # Execute every requested tool call and return each result by tool_call_id
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            raw_args = json.loads(tool_call.function.arguments)
            handler = tool_dispatch.get(fn_name)
            print("#"*50)
            print(fn_name)
            print(raw_args)
            
            if handler is None:
                result = {"error": f"Unknow tool: {fn_name}"}
            else:
                result = handler(raw_args)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
        

In [80]:
answer = ask_with_tool(
    "What is the weather in Bishkek and Tokyo",
    "what is the currenc"
)

##################################################
ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ogpf8grt', function=Function(arguments='{"city":"Bishkek"}', name='get_weather'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_rsj37px5', function=Function(arguments='{"city":"Tokyo"}', name='get_weather'), type='function', index=1)])
[{'role': 'user', 'content': 'What is the weather in Bishkek and Tokyo'}, {'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_ogpf8grt', 'function': {'arguments': '{"city":"Bishkek"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}, {'id': 'call_rsj37px5', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function', 'index': 1}]}]
##################################################

In [74]:
print(answer)

The weather in Bishkek is 55F and shiny. The weather in Tokyo is 32F and rainy.
